# Langmuir oscillations and density inference

Run this notebook from the `langmuir-jax` folder after installing `requirements.txt`. The README explains the linear cold-fluid model, fixed physical units, and limits. No GPU is required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from solver import simulate, field, probe, OMEGA_REF, E_REF
from run_example import validation, infer
from solver import analytical_probe
import jax.numpy as jnp
%matplotlib inline


## Simulate an oscillation
`q` is the equilibrium density divided by the fixed reference density of 10¹⁴ m⁻³.

In [ ]:
q = 1.35
t = np.arange(721) * 0.025
E = np.asarray(probe(q)) * float(E_REF)
plt.plot(t / OMEGA_REF * 1e9, E)
plt.xlabel("Time (ns)")
plt.ylabel("Probe field (V/m)");


## Check the numerical method
This checks frequency convergence, time integration, conservation, Poisson consistency, and automatic derivatives against finite differences.

In [ ]:
checks = validation()


## Infer density from independent synthetic observations
The analytical solution generates the observations. The optimizer evaluates and differentiates the numerical fluid solver. Initial phase, perturbation amplitude, and diagnostic calibration are assumed known.

In [ ]:
data = np.asarray(analytical_probe(q, jnp.asarray(t)))[::4]
fit, history = infer(data)
print(fit)
plt.plot(history[:, 0], "o-")
plt.axhline(q, color="black", linestyle=":")
plt.xlabel("Iteration")
plt.ylabel("Density / reference density");


## Reproduce the published figures and noisy-data experiment
From a terminal run `python run_example.py --output results`. This also writes CSV data, field arrays, an animation, and a JSON validation summary. The noisy case is a single seeded realization, not a confidence interval.